In [0]:
import os

# ==========================
# CONFIG
# ==========================
volume_path = "/Volumes/data_dev_olist/bronze/raw"

storage_account = "quocluudata"
container = "raw-data"

files_to_process = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

# ==========================
# CSV -> BRONZE DELTA
# ==========================
for file_name in files_to_process:

    table_name = file_name.replace("_dataset.csv", "") \
                          .replace(".csv", "")

    source_path = os.path.join(volume_path, file_name)

    target_path = (
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
        f"bronze_delta/{table_name}"
    )

    target_table = f"data_dev_olist.bronze.{table_name}"

    print(f"📥 Reading : {source_path}")
    print(f"📤 Writing : {target_path}")

    try:

        # Read raw csv
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(source_path)
        )

        # Drop old table if exists
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")

        # Write Delta
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", target_path)
            .saveAsTable(target_table)
        )

        print(
            f"✅ {table_name} loaded successfully "
            f"({df.count():,} rows)"
        )

    except Exception as e:
        print(f"❌ Error {table_name}")
        print(str(e))

print("🎉 Bronze Delta completed")

📥 Reading : /Volumes/data_dev_olist/bronze/raw/olist_customers_dataset.csv
📤 Writing : abfss://raw-data@quocluudata.dfs.core.windows.net/bronze_delta/olist_customers
✅ olist_customers loaded successfully (99,441 rows)
📥 Reading : /Volumes/data_dev_olist/bronze/raw/olist_geolocation_dataset.csv
📤 Writing : abfss://raw-data@quocluudata.dfs.core.windows.net/bronze_delta/olist_geolocation
✅ olist_geolocation loaded successfully (1,000,163 rows)
📥 Reading : /Volumes/data_dev_olist/bronze/raw/olist_order_items_dataset.csv
📤 Writing : abfss://raw-data@quocluudata.dfs.core.windows.net/bronze_delta/olist_order_items
✅ olist_order_items loaded successfully (112,650 rows)
📥 Reading : /Volumes/data_dev_olist/bronze/raw/olist_order_payments_dataset.csv
📤 Writing : abfss://raw-data@quocluudata.dfs.core.windows.net/bronze_delta/olist_order_payments
✅ olist_order_payments loaded successfully (103,886 rows)
📥 Reading : /Volumes/data_dev_olist/bronze/raw/olist_order_reviews_dataset.csv
📤 Writing : abfss

In [0]:
%sql
DESCRIBE EXTENDED data_dev_olist.silver.clean_customer;


col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_zip_code_prefix,int,null
customer_city,string,null
customer_state,string,null
is_active,boolean,null
_processed_at,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"is_active, customer_id, customer_state, customer_zip_code_prefix, _processed_at, customer_unique_id, customer_city",
